# Part 2: Model Training & Strategy Formulation (Improved)
In this notebook, we train a heavily improved XGBoost model with:
1. **New cross-sectional features** (rank each asset vs. its peers each day)
2. **Proper early stopping** to prevent overfitting
3. **Aggressive hyperparameter tuning** for noisy financial data

**Prediction Target:** Regression (5-day forward return).
**Model:** XGBoost Regressor.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

data_dir = '/content/drive/MyDrive/precog_data/processed_data'


## 1. Data Loading, Target Definition & New Feature Engineering
We load all 100 assets, define our 5-day forward return target, and then engineer **cross-sectional features** — these rank each asset relative to the other 99 assets on the same day. This is critical because XGBoost needs to know not just "Is RSI high?" but "Is this asset's RSI higher than its peers today?"


In [ ]:
all_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
df_list = []

for file in all_files:
    df = pd.read_csv(file)
    df['Asset'] = os.path.basename(file).replace('.csv', '')
    df['Target_5d'] = df['Ret_5d'].shift(-5)
    df_list.append(df)

full_df = pd.concat(df_list, ignore_index=True)
full_df['Date'] = pd.to_datetime(full_df['Date'])
full_df.sort_values(by=['Date', 'Asset'], inplace=True)

# ============================================================
# NEW: Cross-Sectional Features (rank each asset vs peers)
# ============================================================
# For each day, rank every asset's feature from 0 to 1
# This tells the model: "This asset has the 3rd highest RSI today out of 100"
cross_sectional_cols = ['Ret_1d', 'Ret_5d', 'RSI_14', 'MACD_Hist', 'Dist_SMA_50', 'Vol_20d']

for col in cross_sectional_cols:
    full_df[f'{col}_rank'] = full_df.groupby('Date')[col].rank(pct=True)

# NEW: Mean Reversion Signal — how far is today's return from the asset's own 20-day average?
full_df['Ret_1d_zscore'] = full_df.groupby('Asset')['Ret_1d'].transform(
    lambda x: (x - x.rolling(20).mean()) / x.rolling(20).std()
)

# NEW: Momentum persistence — is the 5-day trend accelerating or decelerating?
full_df['Momentum_accel'] = full_df.groupby('Asset')['Ret_5d'].transform(
    lambda x: x - x.shift(5)
)

# Drop rows with NaNs (from rolling windows and target shift)
full_df.dropna(inplace=True)

print(f"Total dataset size: {full_df.shape}")
print(f"Date range: {full_df['Date'].min()} to {full_df['Date'].max()}")


## 2. Chronological Train/Test Split
80% train / 20% test, split by date to prevent any look-ahead bias.

In [ ]:
dates = full_df['Date'].unique()
split_idx = int(len(dates) * 0.8)
train_dates = dates[:split_idx]
test_dates = dates[split_idx:]

train_df = full_df[full_df['Date'].isin(train_dates)].copy()
test_df = full_df[full_df['Date'].isin(test_dates)].copy()

# EXPANDED feature list (original 14 + 8 new cross-sectional/engineered features)
features = [
    # Original features
    'Ret_1d', 'Ret_5d', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'SMA_10', 'SMA_50', 'Dist_SMA_50', 'BB_Upper', 'BB_Lower',
    'Vol_20d', 'Vol_SMA_20', 'Vol_ROC',
    # NEW: Cross-sectional rank features
    'Ret_1d_rank', 'Ret_5d_rank', 'RSI_14_rank',
    'MACD_Hist_rank', 'Dist_SMA_50_rank', 'Vol_20d_rank',
    # NEW: Engineered features
    'Ret_1d_zscore', 'Momentum_accel'
]
target = 'Target_5d'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples:  {len(X_test):,}")
print(f"Features used:    {len(features)}")


## 3. XGBoost with Early Stopping
Key improvements over the previous version:
- **Early Stopping:** We set `n_estimators` very high (2000) but use `early_stopping_rounds=50`. The model will automatically stop training when performance on the validation set stops improving. This finds the *perfect* number of trees without manually guessing.
- **Lower `max_depth=3`:** Even shallower trees. Financial data is extremely noisy — deeper trees memorize noise.
- **Higher regularization:** We push `reg_alpha` and `reg_lambda` higher to force the model to be conservative and only use features it's truly confident about.
- **`colsample_bytree=0.6`:** Each tree only sees 60% of features, forcing diversity and reducing overfitting.


In [ ]:
model = xgb.XGBRegressor(
    n_estimators=2000,        # High ceiling — early stopping will find the sweet spot
    learning_rate=0.005,      # Very slow learning for subtle pattern detection
    max_depth=3,              # Very shallow trees — critical for noisy financial data
    min_child_weight=100,     # Each leaf must have at least 100 samples (prevents memorizing outliers)
    subsample=0.7,            # Each tree trains on 70% of rows
    colsample_bytree=0.6,    # Each tree sees only 60% of features
    gamma=1.0,                # Minimum loss reduction to make a split (prunes weak splits)
    reg_alpha=1.0,            # L1 regularization
    reg_lambda=5.0,           # L2 regularization (very strong)
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

print("Training XGBoost with early stopping...")
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

# Evaluate
preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"\nTest MSE:  {mse:.6f}")
print(f"Test R^2:  {r2:.6f}")
print(f"Trees used: {model.best_iteration if hasattr(model, 'best_iteration') else 'all'}")


## 4. Feature Importance
Which features does the improved model rely on? The cross-sectional rank features should appear prominently if they're adding value.

In [ ]:
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)

plt.figure(figsize=(12, 7))
colors = ['#2ecc71' if 'rank' in f or 'zscore' in f or 'accel' in f else '#3498db' for f in feat_imp.index]
feat_imp.plot(kind='bar', color=colors)
plt.title('Feature Importance (Green = New Features)', fontsize=14)
plt.ylabel('Relative Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
for f, v in feat_imp.head(5).items():
    print(f"  {f}: {v:.4f}")


## 5. Generate Signals on Out-of-Sample Data
Same signal logic as before: Top 20% predicted returns → Long (+1), Bottom 20% → Short (-1). Saved directly to Google Drive for Part 3.

In [ ]:
test_df['Predicted_Ret_5d'] = preds

def generate_signals(group):
    upper_thresh = group['Predicted_Ret_5d'].quantile(0.80)
    lower_thresh = group['Predicted_Ret_5d'].quantile(0.20)
    
    conditions = [
        (group['Predicted_Ret_5d'] > upper_thresh),
        (group['Predicted_Ret_5d'] < lower_thresh)
    ]
    choices = [1, -1]
    
    group['Signal'] = np.select(conditions, choices, default=0)
    return group

print("Generating signals on Out-Of-Sample Data...")
test_df = test_df.groupby('Date').apply(generate_signals)

print(test_df['Signal'].value_counts(normalize=True))

# Save directly to Google Drive
file_path = '/content/drive/MyDrive/signals.csv'
test_df[['Date', 'Asset', 'Close', 'Target_5d', 'Predicted_Ret_5d', 'Signal']].to_csv(file_path, index=False)
print(f"\nSignals saved successfully to: {file_path}")
